# Sky Object Detector — YOLOv11x (Kaggle T4 x2)
**6 Classes:** drone · bird · airplane · helicopter · balloon · kite

| Class | Cap | Class | Cap |
|-------|-----|-------|-----|
| drone | 3,000 | helicopter | 2,000 |
| bird | 2,500 | balloon | 1,000 |
| airplane | 2,000 | kite | ~313 |
| background negatives | 500 | **TOTAL** | **~12,000** |

---
### BEFORE RUNNING:
1. Right panel → **Accelerator: GPU T4 x2**
2. Right panel → **Internet: ON**
3. Run all cells top to bottom

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Install & verify GPU
# ════════════════════════════════════════════════════════════
!pip install ultralytics roboflow pyyaml -q

import torch
from pathlib import Path

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
n_gpu = torch.cuda.device_count()
print(f'GPUs    : {n_gpu}')
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {props.name}  {props.total_memory // 1024**3} GB')

assert torch.cuda.is_available(), 'ERROR: No GPU — enable T4 x2 in Session options'

DEVICE = '0,1' if n_gpu >= 2 else '0'
BATCH  = 32    if n_gpu >= 2 else 16
print(f'\n>>> Device={DEVICE}  Batch={BATCH}  — ready')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Download YOUR Roboflow dataset (3D printed drones)
# ════════════════════════════════════════════════════════════
from roboflow import Roboflow

API_KEY = 'ZaOMj2jVJyUuhkIiLYad'
rf      = Roboflow(api_key=API_KEY)
project = rf.workspace('fadils-workspace').project('drone-detection-puymg')

versions = project.versions()
latest   = max(versions, key=lambda v: v.version)
print(f'Downloading v{latest.version}...')
latest.download('yolov8', location='/kaggle/working/ds_custom')
print('Custom 3D drones ✓')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Download Kaggle datasets
# ════════════════════════════════════════════════════════════
import subprocess

def kaggle_dl(slug, dest):
    r = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', slug, '-p', dest, '--unzip'],
        capture_output=True, text=True
    )
    status = '✓' if r.returncode == 0 else f'✗  {r.stderr.strip()[:80]}'
    print(f'  {status}  {slug}')
    return r.returncode == 0

print('── Drones ──')
kaggle_dl('muki2003/yolo-drone-detection-dataset',  '/kaggle/working/ds_drones_out')
kaggle_dl('sshikamaru/drone-yolo-detection',        '/kaggle/working/ds_drones_ex')

print('── Airplane / Military ──')
kaggle_dl('rookieengg/military-aircraft-detection-dataset-yolo-format', '/kaggle/working/ds_mil_k')

print('── Balloon ──')
kaggle_dl('vbookshelf/v2-balloon-detection-dataset', '/kaggle/working/ds_balloon')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Download Roboflow Universe public datasets
# ════════════════════════════════════════════════════════════
from roboflow import Roboflow

API_KEY = 'ZaOMj2jVJyUuhkIiLYad'
rf = Roboflow(api_key=API_KEY)

RF_DATASETS = [
    # (workspace,              project,                        dest,                       description)
    ('yolo-9evjx',             'birds-wijmc',                  '/kaggle/working/ds_birds',  'Birds 4658 imgs'),
    ('yolo-7zf46',             'airplanes-ecccl',              '/kaggle/working/ds_planes', 'Airplanes 3324 imgs'),
    ('missile-thingie',        'fighter-jet-detection',        '/kaggle/working/ds_jets',   'Fighter Jets 10100 imgs (43 types)'),
    ('rl4pcd',                 'yolo-military-s48o9',          '/kaggle/working/ds_mil_rf', 'Military multi-class'),
    ('helicoptersofdc',        'helicopters-of-dc-ghwuq',      '/kaggle/working/ds_heli',   'Helicopters 5483 imgs'),
    ('lenn-van-genechten-gjgla','kite-detection-v2',           '/kaggle/working/ds_kite',   'Kites 313 imgs'),
    ('mini-guardian',          'balloon-detection-y4tjv',      '/kaggle/working/ds_bal_rf', 'Balloons RF'),
]

for ws, proj, dest, desc in RF_DATASETS:
    try:
        p = rf.workspace(ws).project(proj)
        vs = p.versions()
        v  = max(vs, key=lambda x: x.version)
        v.download('yolov8', location=dest)
        print(f'  ✓  {desc}')
    except Exception as e:
        print(f'  ✗  {desc}  →  {str(e)[:80]}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Merge with class remapping + per-class image caps
#
# Classes: 0=drone  1=bird  2=airplane  3=helicopter
#          4=balloon  5=kite
# Caps:    drone=3000  bird=2500  airplane=2000  helicopter=2000
#          balloon=1000  kite=all(~313)  background=500
# ════════════════════════════════════════════════════════════
import shutil, yaml
from pathlib import Path
from collections import Counter

# ── Config ───────────────────────────────────────────────────
CLASSES = ['drone', 'bird', 'airplane', 'helicopter', 'balloon', 'kite']

CLASS_CAPS = {
    0: 3000,   # drone
    1: 2500,   # bird
    2: 2000,   # airplane (commercial + all military/jets)
    3: 2000,   # helicopter
    4: 1000,   # balloon
    5: 9999,   # kite — take all (~313 available)
}
BG_CAP = 500

# Name → class ID (handles any variation in dataset class names)
NAME_TO_CLS = {
    # drone
    'drone':0,'uav':0,'quadcopter':0,'multirotor':0,'copter':0,'dji':0,'fpv':0,
    # bird
    'bird':1,'birds':1,'seagull':1,'pigeon':1,'sparrow':1,'eagle':1,'hawk':1,
    # airplane — ALL aircraft types map to class 2
    'airplane':2,'aeroplane':2,'plane':2,'aircraft':2,'jet':2,
    'a10':2,'a-10':2,'b1':2,'b-1':2,'b2':2,'b-2':2,'b52':2,'b-52':2,
    'f16':2,'f-16':2,'f22':2,'f-22':2,'f35':2,'f-35':2,
    'sr71':2,'sr-71':2,'u2':2,'c130':2,'c-130':2,'kc135':2,
    'su57':2,'su-57':2,'mig29':2,'eurofighter':2,'rafale':2,'typhoon':2,
    'fighter':2,'bomber':2,'airliner':2,'commercial':2,'military-aircraft':2,
    # helicopter
    'helicopter':3,'heli':3,'chopper':3,'rotorcraft':3,
    'uh-60':3,'ch-47':3,'ah-64':3,'blackhawk':3,'apache':3,'chinook':3,
    # balloon
    'balloon':4,'hot-air-balloon':4,'hot_air_balloon':4,'blimp':4,'airship':4,
    # kite
    'kite':5,'kites':5,
}

# ── Helpers ──────────────────────────────────────────────────
class_img_counts = Counter()   # images added per class

def read_remap(dataset_dir):
    for yf in sorted(Path(dataset_dir).rglob('*.yaml')):
        try:
            info = yaml.safe_load(open(yf))
            names = info.get('names', [])
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names.keys())]
            remap = {}
            for i, n in enumerate(names):
                key = n.lower().strip().replace(' ','-').replace('_','-')
                if key in NAME_TO_CLS:
                    remap[i] = NAME_TO_CLS[key]
                elif key.replace('-','') in NAME_TO_CLS:
                    remap[i] = NAME_TO_CLS[key.replace('-','')]
            print(f'    yaml={yf.name} | classes={names} | remap={remap}')
            return remap
        except:
            pass
    return {}

def to_bbox(parts, cls_id):
    v = list(map(float, parts[1:]))
    if len(v) == 4:
        return f'{cls_id} {v[0]:.6f} {v[1]:.6f} {v[2]:.6f} {v[3]:.6f}'
    if len(v) >= 6 and len(v) % 2 == 0:
        xs=v[0::2]; ys=v[1::2]
        xc=(min(xs)+max(xs))/2; yc=(min(ys)+max(ys))/2
        w=max(xs)-min(xs); h=max(ys)-min(ys)
        return (f'{cls_id} {max(0.,min(1.,xc)):.6f} {max(0.,min(1.,yc)):.6f} '
                f'{max(0.001,min(1.,w)):.6f} {max(0.001,min(1.,h)):.6f}')
    return None

def copy_ds(img_dir, lbl_dir, dst_split, tag, remap, force_cls=None):
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    if not img_dir.exists():
        return 0
    copied = 0
    for img in sorted(img_dir.glob('*')):
        if img.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp'}:
            continue
        lbl = lbl_dir / (img.stem + '.txt')
        raw = lbl.read_text().strip().splitlines() if lbl.exists() else []

        new_lines = []
        primary = None

        if not raw:
            # Background negative image
            if class_img_counts['bg'] >= BG_CAP:
                continue
            class_img_counts['bg'] += 1
            primary = 'bg'
        else:
            for line in raw:
                parts = line.strip().split()
                if len(parts) < 5: continue
                orig = int(parts[0])
                nc = force_cls if force_cls is not None else remap.get(orig)
                if nc is None: continue
                out = to_bbox(parts, nc)
                if out: new_lines.append((nc, out))
            if not new_lines:
                continue
            # Use the most frequent class in this image as primary
            cls_freq = Counter(nc for nc, _ in new_lines)
            primary = cls_freq.most_common(1)[0][0]
            # Check cap for primary class
            if class_img_counts[primary] >= CLASS_CAPS.get(primary, 9999):
                continue
            class_img_counts[primary] += 1

        shutil.copy(img, MERGED / dst_split / 'images' / f'{tag}_{img.name}')
        dst_lbl = MERGED / dst_split / 'labels' / f'{tag}_{img.stem}.txt'
        dst_lbl.write_text('\n'.join(line for _, line in new_lines))
        copied += 1
    return copied

def add_ds(base_dir, tag, force_cls=None, use_remap=False):
    base = Path(base_dir)
    if not base.exists():
        print(f'  SKIP (missing): {base_dir}')
        return 0
    remap = read_remap(base) if use_remap else {}
    img_dirs = [p for p in base.rglob('images') if p.is_dir()] or [base]
    total = 0
    for img_dir in img_dirs:
        parent = img_dir.parent.name.lower()
        dst = 'valid' if any(x in parent for x in ('val','test','valid')) else 'train'
        lbl = img_dir.parent/'labels'
        if not lbl.exists(): lbl = img_dir
        n = copy_ds(img_dir, lbl, dst, tag, remap, force_cls)
        if n: print(f'    [{dst}] {n} imgs')
        total += n
    return total

# ── Build merge directories ───────────────────────────────────
MERGED = Path('/kaggle/working/merged')
for s in ['train','valid']:
    (MERGED/s/'images').mkdir(parents=True, exist_ok=True)
    (MERGED/s/'labels').mkdir(parents=True, exist_ok=True)

# ════ DRONE (class 0, cap 3000) ══════════════════════════════
print('\n── Drone (cap=3000) ──')
CUSTOM = Path('/kaggle/working/ds_custom')
for sn, dst in [('train','train'),('valid','valid')]:
    id_ = CUSTOM/sn/'images'
    if not id_.exists(): id_ = CUSTOM/sn
    ld_ = CUSTOM/sn/'labels'
    if not ld_.exists(): ld_ = id_
    n = copy_ds(id_, ld_, dst, 'cust', {}, force_cls=0)
    if n: print(f'    custom [{dst}]: {n}')
add_ds('/kaggle/working/ds_drones_out', 'dout',  force_cls=0)
add_ds('/kaggle/working/ds_drones_ex',  'dex',   force_cls=0)
print(f'  → drone total so far: {class_img_counts[0]}')

# ════ BIRD (class 1, cap 2500) ═══════════════════════════════
print('\n── Bird (cap=2500) ──')
add_ds('/kaggle/working/ds_birds', 'bird', force_cls=1)
print(f'  → bird total: {class_img_counts[1]}')

# ════ AIRPLANE (class 2, cap 2000) ═══════════════════════════
print('\n── Airplane (cap=2000, all aircraft types → class 2) ──')
add_ds('/kaggle/working/ds_planes',  'ap',    force_cls=2)
add_ds('/kaggle/working/ds_jets',    'jet',   force_cls=2)   # fighter jets (43 types)
add_ds('/kaggle/working/ds_mil_k',   'mil_k', force_cls=2)   # Kaggle military
add_ds('/kaggle/working/ds_mil_rf',  'mil_r', use_remap=True) # Roboflow military (multi-class)
print(f'  → airplane total: {class_img_counts[2]}')

# ════ HELICOPTER (class 3, cap 2000) ═════════════════════════
print('\n── Helicopter (cap=2000) ──')
add_ds('/kaggle/working/ds_heli', 'heli', force_cls=3)
print(f'  → helicopter total: {class_img_counts[3]}')

# ════ BALLOON (class 4, cap 1000) ════════════════════════════
print('\n── Balloon (cap=1000) ──')
add_ds('/kaggle/working/ds_balloon', 'bal',    force_cls=4)
add_ds('/kaggle/working/ds_bal_rf',  'bal_rf', force_cls=4)
print(f'  → balloon total: {class_img_counts[4]}')

# ════ KITE (class 5, all ~313) ════════════════════════════════
print('\n── Kite (take all ~313) ──')
add_ds('/kaggle/working/ds_kite', 'kite', force_cls=5)
print(f'  → kite total: {class_img_counts[5]}')

# ════ SUMMARY ════════════════════════════════════════════════
tr = len(list((MERGED/'train'/'images').glob('*')))
va = len(list((MERGED/'valid'/'images').glob('*')))
print(f'\n{"="*50}')
print(f'MERGED  train={tr}  valid={va}  TOTAL={tr+va}')
print(f'Background negatives: {class_img_counts["bg"]}')
print('\nPer-class image counts:')
for i, name in enumerate(CLASSES):
    cap = CLASS_CAPS[i]
    got = class_img_counts[i]
    bar = '█' * int(got/cap*20) if cap < 9999 else '█' * min(20, got//15)
    print(f'  [{i}] {name:12s}  {got:5d} / {cap if cap<9999 else "all":>4}  {bar}')

# Write data.yaml
yaml_txt = f"""train: /kaggle/working/merged/train/images
val:   /kaggle/working/merged/valid/images
nc: {len(CLASSES)}
names: {CLASSES}
"""
Path('/kaggle/working/merged/data.yaml').write_text(yaml_txt)
print('\ndata.yaml written ✓')
print(yaml_txt)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Download pre-trained YOLOv11x drone weights
# ════════════════════════════════════════════════════════════
from pathlib import Path

!wget -q --show-progress \
    'https://huggingface.co/doguilmak/Drone-Detection-YOLOv11x/resolve/main/weight/best.pt?download=true' \
    -O /kaggle/working/pretrained.pt

sz = Path('/kaggle/working/pretrained.pt').stat().st_size / 1e6
assert sz > 50, f'Download failed — file too small: {sz:.1f} MB'
print(f'Pretrained weights: {sz:.1f} MB  ✓')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — TRAIN
# ════════════════════════════════════════════════════════════
from ultralytics import YOLO

model = YOLO('/kaggle/working/pretrained.pt')

model.train(
    data          = '/kaggle/working/merged/data.yaml',
    imgsz         = 640,
    batch         = BATCH,
    epochs        = 80,
    patience      = 15,

    device        = DEVICE,
    workers       = 4,

    # Head re-initialised for 6 classes — freeze fewer backbone layers
    freeze        = 5,
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    close_mosaic  = 15,

    # Augmentation
    degrees       = 45,
    translate     = 0.1,
    scale         = 0.5,
    fliplr        = 0.5,
    flipud        = 0.2,
    mosaic        = 1.0,
    mixup         = 0.15,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    erasing       = 0.3,

    project       = '/kaggle/working/runs',
    name          = 'sky_detector',
    exist_ok      = True,
    save          = True,
    save_period   = 10,
    plots         = True,
    verbose       = True,
)

print('\n' + '='*50)
print('TRAINING COMPLETE')
print('='*50)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Validate
# ════════════════════════════════════════════════════════════
from ultralytics import YOLO

best = YOLO('/kaggle/working/runs/sky_detector/weights/best.pt')
m = best.val(data='/kaggle/working/merged/data.yaml', imgsz=640, verbose=False)

classes = ['drone','bird','airplane','helicopter','balloon','kite']
print('\n════ FINAL MODEL METRICS ════')
print(f'  mAP@50     : {m.box.map50:.4f}')
print(f'  mAP@50-95  : {m.box.map:.4f}')
print(f'  Precision  : {m.box.mp:.4f}')
print(f'  Recall     : {m.box.mr:.4f}')
print('\nPer-class AP@50:')
for i, ap in enumerate(m.box.ap50):
    print(f'  [{i}] {classes[i]:12s}  {ap:.4f}')
print('════════════════════════════')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — Training curves
# ════════════════════════════════════════════════════════════
from IPython.display import Image, display
from pathlib import Path

run = Path('/kaggle/working/runs/sky_detector')
for fname in ['results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    p = run / fname
    if not p.exists(): p = run / fname.replace('_normalized','')
    if p.exists():
        print(f'\n── {fname} ──')
        display(Image(str(p), width=900))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — Visual test (sample from each class)
# ════════════════════════════════════════════════════════════
from ultralytics import YOLO
from IPython.display import Image, display
from pathlib import Path
import random

best = YOLO('/kaggle/working/runs/sky_detector/weights/best.pt')

TAG_MAP = {
    'cust':'DRONE-3D','dout':'DRONE-OUT','dex':'DRONE-EX',
    'bird':'BIRD','ap':'AIRPLANE','jet':'FIGHTER-JET',
    'heli':'HELICOPTER','bal':'BALLOON','kite':'KITE',
}

val_imgs = list(Path('/kaggle/working/merged/valid/images').glob('*'))
sample   = random.sample(val_imgs, min(12, len(val_imgs)))
out_dir  = Path('/kaggle/working/test_preds')
out_dir.mkdir(exist_ok=True)

for img_path in sample:
    best.predict(str(img_path), conf=0.30, save=True,
                 project=str(out_dir), name='run', exist_ok=True, verbose=False)
    saved = sorted((out_dir/'run').glob('*.jpg'), key=lambda p: p.stat().st_mtime)
    if saved:
        prefix = img_path.name.split('_')[0]
        tag = TAG_MAP.get(prefix, prefix.upper())
        print(f'[{tag}] {img_path.name}')
        display(Image(str(saved[-1]), width=640))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — Export to NCNN (Raspberry Pi)
# ════════════════════════════════════════════════════════════
from ultralytics import YOLO

best = YOLO('/kaggle/working/runs/sky_detector/weights/best.pt')
best.export(format='ncnn', imgsz=640)
print('NCNN export done.')
!ls /kaggle/working/runs/sky_detector/weights/

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — Zip best.pt + NCNN → download
# ════════════════════════════════════════════════════════════
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

weights = Path('/kaggle/working/runs/sky_detector/weights')

with zipfile.ZipFile('/kaggle/working/sky_best.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(weights/'best.pt', 'best.pt')
    ncnn = weights/'best_ncnn_model'
    if ncnn.exists():
        for f in ncnn.rglob('*'):
            if f.is_file():
                zf.write(f, f'best_ncnn_model/{f.name}')

sz = Path('/kaggle/working/sky_best.zip').stat().st_size / 1e6
print(f'sky_best.zip  →  {sz:.1f} MB')
print('Contains: best.pt + best_ncnn_model/')
print('\n>>> Click to download <<<')
display(FileLink('/kaggle/working/sky_best.zip'))

---
## After downloading `sky_best.zip`
Extract → put `best.pt` in `C:\Users\Fadil Laptop\Desktop\Drone\`

**Test on PC:**
```python
from ultralytics import YOLO
model = YOLO('best.pt')
model.predict('test.jpg', conf=0.35, show=True)
```

**Pi with NCNN:** copy `best_ncnn_model/` folder → `MODEL_PATH = 'best_ncnn_model'`

### Class IDs
| ID | Class | ID | Class |
|----|-------|----|-------|
| 0 | drone | 3 | helicopter |
| 1 | bird | 4 | balloon |
| 2 | airplane | 5 | kite |